# Tests

In [2]:
from our_pipeline.search_tools import LawSearchTool, DenseSearchTool
from our_pipeline.dense.index import DenseIndex
from our_pipeline.bm25.corpus import get_or_build_index
from our_pipeline.constants import (
    CONFIG,
    FORCE_REBUILD_INDICES,
    LAWS_CSV,
    LAWS_DENSE_META_PATH,
    LAWS_FAISS_PATH,
    LAWS_INDEX_PATH,
    OUTPUT_PATH,
    DATA_PATH
)
import re
import pandas as pd

Loading API model: qwen3.5-27b
API model client loaded successfully!


### Read val_df and submissions_df

In [3]:
val_df = pd.read_csv(f'{DATA_PATH}/val.csv')

In [4]:
just_law = True
submission_path = OUTPUT_PATH / "submission13-06_11-16.csv"
submission_df = pd.read_csv(submission_path)

### Check length of citations list in submission_df

In [5]:
submission_df['len'] = submission_df['predicted_citations'].apply(lambda x: len(x.split(';')))
print(submission_df['len'])

0    157
1    151
2    158
3    154
4    159
5    157
6    154
7    158
8    158
9    160
Name: len, dtype: int64


### Recall for every query

In [6]:
for j in range(10):
    ith_row = val_df.iloc[j]
    query = ith_row['query']
    y_true = ith_row['gold_citations'].split(';')
    if just_law:
        y_true = list(filter(lambda x: x[0] == 'A', y_true))
    y_pred = submission_df.iloc[j]['predicted_citations'].split(';')

    correct = [i for i in y_pred if i in y_true]
    print(f'\n########### val00{j}')
    print(f"Found {len(correct)} out of {len(y_true)}")
    print('recall:', len(correct) / len(y_true))


########### val000
Found 6 out of 19
recall: 0.3157894736842105

########### val001
Found 3 out of 20
recall: 0.15

########### val002
Found 2 out of 24
recall: 0.08333333333333333

########### val003
Found 6 out of 9
recall: 0.6666666666666666

########### val004
Found 2 out of 6
recall: 0.3333333333333333

########### val005
Found 3 out of 11
recall: 0.2727272727272727

########### val006
Found 3 out of 15
recall: 0.2

########### val007
Found 1 out of 20
recall: 0.05

########### val008
Found 2 out of 11
recall: 0.18181818181818182

########### val009
Found 1 out of 14
recall: 0.07142857142857142


## Search Tools

### BM25 Law

In [ ]:
laws_index = get_or_build_index(
    name="laws",
    csv_path=LAWS_CSV,
    index_path=LAWS_INDEX_PATH,
    force_rebuild=FORCE_REBUILD_INDICES,
    # max_rows=10000  # Uncomment to test with smaller corpus
)

# Create tools
law_tool = LawSearchTool(
    index=laws_index,
    top_k=CONFIG["top_k_laws"],
    max_excerpt_length=300,
)

### Dense Embeddings Law

In [9]:
_description = """Dense search over Swiss federal laws"""

dense_law_tool = DenseSearchTool(
            index=DenseIndex.load(LAWS_FAISS_PATH, LAWS_DENSE_META_PATH),
            name="dense_search_laws",
            description=_description,
            top_k=CONFIG["top_k_dense_laws"],
            max_excerpt_length=300,
        )

## Search one query

In [10]:
index_to_check = 0
ith_row = val_df.iloc[index_to_check]
query = ith_row['query']
y_true = ith_row['gold_citations'].split(';')
if just_law:
    y_true = list(filter(lambda x: x[0] == 'A', y_true))
result = dense_law_tool(query)
y_pred = re.findall(r'-\s*(.+?)\s*:', result)

correct = [i for i in y_pred if i in y_true]
print(f'\n########### val00{index_to_check}')
print(f"Found {len(correct)} out of {len(y_true)}")
print('recall:', len(correct) / len(y_true))

The `tokenizer_kwargs` argument was renamed and is now deprecated. Please use `processor_kwargs` instead.


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]


########### val000
Found 5 out of 19
recall: 0.2631578947368421
